In [3]:
import os
import faiss
import numpy as np
from sentence_transformers import SentenceTransformer
import ollama

In [8]:
# 参数
DATA_DIR = "rag_data"
CHUNK_SIZE = 512
CHUNK_OVERLAP = 50
TOP_K = 3
EMBED_MODEL = "all-MiniLM-L6-v2"
LLM_MODEL = "qwen2.5:7b"

In [5]:
def load_documents(data_dir):
    texts = []
    for filename in os.listdir(data_dir):
        if filename.endswith(".txt"):
            with open(os.path.join(data_dir, filename), "r", encoding="utf-8") as f:
                texts.append(f.read())
    return texts


def chunk_text(text, chunk_size, overlap):
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunks.append(text[start:end])
        start += chunk_size - overlap
    return chunks


def build_index(chunks, model):
    embeddings = model.encode(chunks, show_progress_bar=True)
    embeddings = np.array(embeddings).astype("float32")

    dim = embeddings.shape[1]
    index = faiss.IndexFlatL2(dim)
    index.add(embeddings)

    return index, embeddings


def retrieve(query, model, index, chunks, top_k):
    query_vec = model.encode([query]).astype("float32")
    distances, indices = index.search(query_vec, top_k)

    results = []
    for idx in indices[0]:
        results.append(chunks[idx])

    return results


def generate_answer(query, contexts):
    context_text = "\n\n".join(contexts)

    prompt = f"""
You are a helpful assistant.
Answer the question using ONLY the provided context.
If the answer is not in the context, say "I don't know."

Context:
{context_text}

Question:
{query}

Answer:
"""

    response = ollama.chat(
        model=LLM_MODEL,
        messages=[{"role": "user", "content": prompt}]
    )

    return response["message"]["content"]

In [6]:
def main():
    print("Loading documents...")
    docs = load_documents(DATA_DIR)

    print("Chunking...")
    chunks = []
    for doc in docs:
        chunks.extend(chunk_text(doc, CHUNK_SIZE, CHUNK_OVERLAP))

    print(f"Total chunks: {len(chunks)}")

    print("Loading embedding model...")
    embed_model = SentenceTransformer(EMBED_MODEL)

    print("Building FAISS index...")
    index, _ = build_index(chunks, embed_model)

    print("RAG system ready.\n")

    while True:
        query = input("Enter your question (or 'exit'): ")
        if query.lower() == "exit":
            break

        contexts = retrieve(query, embed_model, index, chunks, TOP_K)

        print("\n=== Retrieved Context ===")
        for i, c in enumerate(contexts):
            print(f"\n--- Chunk {i+1} ---\n{c[:300]}")

        answer = generate_answer(query, contexts)

        print("\n=== Answer ===")
        print(answer)
        print("\n" + "="*50 + "\n")

In [10]:
main()

Loading documents...
Chunking...
Total chunks: 59
Loading embedding model...


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 903.67it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Building FAISS index...


Batches: 100%|██████████| 2/2 [00:00<00:00,  2.68it/s]

RAG system ready.




=== Retrieved Context ===

--- Chunk 1 ---
基于DLinear模型的电量预测研究

一、背景
　　　随着经济社会发展，电力需求不断增长，实现对电力负荷的高精度预测具有重要意义。传统方法在处理非平稳、具有趋势和季节性的电量数据时往往精度有限。近年来，基于分解思想的DLinear模型在时间序列预测任务中表现优异。该模型通过对时间序列数据进行趋势与季节成分分解，并分别用单层线性模块拟合，显著提升了可解释性和泛化能力。另外，再引入RevIN模块，进行归一化与反归一化操作，提升模型的预测性能。
二、数据介绍
　　　选取陕西省一般工商业2023年1月1日至2025年4月13日共834天的每日用电量数据。下图1展示了用电量的变化趋势情况，整体用电量值

--- Chunk 2 ---
和残差（Residual）三部分，如下图20所示。趋势成分显示整体用电量呈现缓慢下降态势，符合历史4-5月用电量变化总体情况；季节成分则展现出显著的7天周期性波动，符合工作日和周末的用电差异；残差部分波动较小，说明模型预测中未能解释的成分相对较少。

图 20 DLinear预测结果成分分解
五、结论
　　本研究基于DLinear模型成功实现了对陕西一般工商业用电量的精准建模与预测。DLinear通过将时间序列分解为趋势项与季节项，并分别使用单层线性模块拟合，实现了高效、稳定且具备良好可解释性的负荷预测结果。同时，引入RevIN模块解决非平稳性问题，进一步提高了模型的鲁棒性和泛化能力。整体预测

--- Chunk 3 ---
到层的索引。在嵌入层（Embedding）中和投影层（Projection）中均是由多层感知机（Multilayer Perceptron，MLP）实现的。获得的每个变量token通过自注意力机制相互作用，揭示变量之间的内在关系。并由每个TrmBlock中共享的前馈神经网络独立处理，有助于更好的捕捉各token之间的跨时间关系，由于序列顺序信息隐式存储在前馈神经网络的神经元排列中，因此不再需要传统Transformer结构中的位置编码模块。

图 21 iTransformer算法流程图
3.2.3 小结
　　作为一种面向时间序列预测任务的结构创新，iTransformer保留了传统Trans

=== Answer ===
DLi

KeyboardInterrupt: Interrupted by user